In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load datasets
X = pd.read_excel('minmax.xlsx')
y = pd.read_csv('idC_with_header.csv')['Label']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define fitness function
def fitness_function(params, model_type='svm'):
    if model_type == 'svm':
        C, gamma = params
        model = SVC(C=C, gamma=gamma)
    elif model_type == 'rf':
        n_estimators, max_depth = params
        model = RandomForestClassifier(n_estimators=int(n_estimators), max_depth=int(max_depth))
    else:
        raise ValueError("Unsupported model type.")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return accuracy_score(y_test, predictions)

# GSA parameters
num_agents = 10
max_iter = 20
G0 = 100
alpha = 20

# Hyperparameter bounds
svm_bounds = {'C': (0.1, 100), 'gamma': (0.0001, 1)}
rf_bounds = {'n_estimators': (10, 200), 'max_depth': (1, 50)}

# Initialize agents
def initialize_agents(bounds):
    agents = []
    for _ in range(num_agents):
        agent = []
        for key in bounds:
            low, high = bounds[key]
            agent.append(np.random.uniform(low, high))
        agents.append(agent)
    return np.array(agents)

# GSA optimization
def gsa_optimization(bounds, model_type='svm'):
    agents = initialize_agents(bounds)
    velocities = np.zeros_like(agents)
    best_agent = None
    best_fitness = -np.inf

    for t in range(max_iter):
        fitness = np.array([fitness_function(agent, model_type) for agent in agents])
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_fitness:
            best_fitness = fitness[best_idx]
            best_agent = agents[best_idx].copy()

        worst = np.min(fitness)
        best = np.max(fitness)
        mass = (fitness - worst) / (best - worst + 1e-10)
        mass = mass / mass.sum()

        G = G0 * np.exp(-alpha * t / max_iter)

        for i in range(num_agents):
            force = np.zeros(agents.shape[1])
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(agents[j] - agents[i]) + 1e-10
                    force += np.random.rand() * G * (mass[i] * mass[j]) * (agents[j] - agents[i]) / distance
            acceleration = force / (mass[i] + 1e-10)
            velocities[i] = np.random.rand() * velocities[i] + acceleration
            agents[i] += velocities[i]

            # Ensure agents stay within bounds
            for k, key in enumerate(bounds):
                low, high = bounds[key]
                agents[i][k] = np.clip(agents[i][k], low, high)

    return best_agent

# Optimize SVM
best_svm_params = gsa_optimization(svm_bounds, model_type='svm')
best_C, best_gamma = best_svm_params
svm_model = SVC(C=best_C, gamma=best_gamma)
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

# Evaluate SVM
print("SVM Performance:")
print("Accuracy:", accuracy_score(y_test, svm_predictions))
print("Precision:", precision_score(y_test, svm_predictions, average='weighted'))
print("Recall:", recall_score(y_test, svm_predictions, average='weighted'))
print("F1 Score:", f1_score(y_test, svm_predictions, average='weighted'))
print("Best Hyperparameters:", {'C': best_C, 'gamma': best_gamma})

# Optimize RF
best_rf_params = gsa_optimization(rf_bounds, model_type='rf')
best_n_estimators, best_max_depth = best_rf_params
rf_model = RandomForestClassifier(n_estimators=int(best_n_estimators), max_depth=int(best_max_depth))
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# Evaluate RF
print("\nRandom Forest Performance:")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("Precision:", precision_score(y_test, rf_predictions, average='weighted'))
print("Recall:", recall_score(y_test, rf_predictions, average='weighted'))
print("F1 Score:", f1_score(y_test, rf_predictions, average='weighted'))
print("Best Hyperparameters:", {'n_estimators': int(best_n_estimators), 'max_depth': int(best_max_depth)})


SVM Performance:
Accuracy: 0.9662921348314607
Precision: 0.9674157303370786
Recall: 0.9662921348314607
F1 Score: 0.9660770926294285
Best Hyperparameters: {'C': 99.99914054041716, 'gamma': 0.00010090060384349631}

Random Forest Performance:
Accuracy: 0.9325842696629213
Precision: 0.9468164794007491
Recall: 0.9325842696629213
F1 Score: 0.9335713960787881
Best Hyperparameters: {'n_estimators': 105, 'max_depth': 42}
